In [15]:
import sc2reader
import pandas as pd
import numpy as np
import tqdm
from utils import *
from glob import glob

game_number = 0
files = glob('../data/input/pro_data/*')
all_dfs = []
for file in tqdm.tqdm(files):
    game_number += 1
    try:
        replay = sc2reader.load_replay(file, load_map=True, load_level=4)
    except Exception as e:
        print(f"Failed to load {file}: {e}")
        continue
    
    assert replay.map_name in ['valid_maps', "At Eternity's Edge LE", 'Blackrock LE', 'Fear and Faith LE', 'Rainfall LE', 'Sanctuary III LE', 'Lockdown LE', 'Washout LE', 'Rorschach LE', 'Old Sun Temple LE'], replay.map_name

    is_valid_release = replay.release_string >= '5.0.16'
    assert is_valid_release, replay.release_string
    
    player1 = 'Serral'
    player2 = None
    player1_won = None
    player1_race = 'Zerg'
    player2_race = None
    for player in replay.players:
        if player.name == 'Serral':
            assert player.play_race == 'Zerg'
            player1_won = player.result
        else:
            player2 = player.name
            player2_race = player.play_race
    print(player1, player2, player1_race, player2_race)
    
    larva_amounts_data = []
    larva_data = []
    current_larva_count = 0
    for event in replay.tracker_events:
        seconds = event.frame / 22.4
        minutes = int(seconds // 60)
        remaining_seconds = int(seconds % 60)
        if seconds < 1:
            continue
    
        try: 
            if player2 in str(event):
                continue           
            if event.player == player2:
                continue
        except:
            pass
        
        print(seconds, event)
        if "changed to Egg" in str(event):
            current_larva_count = max(0, current_larva_count - 1)
        if "Unit born Larva" in str(event):
            current_larva_count += 1            
            larva_data.append(seconds)
            larva_amounts_data.append(current_larva_count)
        if seconds > 60*10:
            break
    output_df = pd.DataFrame()
    output_df['larva_time'] = larva_data
    output_df['larva_amount'] = larva_amounts_data
    output_df['game_number'] = game_number
    output_df['opponent_race'] = player2_race
    output_df['game_length_seconds_max_600'] = int(np.round(seconds,0))
    output_df['main_player_won'] = player1_won
    output_df = output_df[['game_number', 'opponent_race', 'game_length_seconds_max_600', 'larva_time', 'larva_amount']]
    all_dfs.append(output_df)
    break
output_df = pd.concat(all_dfs, axis='index', ignore_index=True)
# local.write.csv(output_df, '6_extract_pro_larva')
print('done')

  0%|          | 0/31 [00:00<?, ?it/s]

Serral Geralt Zerg Protoss
7.142857142857143 00.10	 Player 1 - Serral (Zerg) - Stats Update
9.910714285714286 00.13	 Player 1 - Serral (Zerg) - Unit Larva [3740001] type changed to Egg
10.223214285714286 00.14	 Player 1 - Serral (Zerg) - Unit born Larva [3C00001]
12.455357142857144 00.17	 Player 1 - Serral (Zerg) - Unit born Drone [3C40001]
12.455357142857144 00.17	 Player 1 - Serral (Zerg) - Unit Larva [36C0001] type changed to Larva
12.455357142857144 00.17	 Player 1 - Serral (Zerg) - Unit died Larva [36C0001].
14.285714285714286 00.20	 Player 1 - Serral (Zerg) - Stats Update
16.16071428571429 00.22	 Player 1 - Serral (Zerg) - Unit Larva [3700001] type changed to Egg
20.133928571428573 00.28	 Player 1 - Serral (Zerg) - Unit born Larva [3C80001]
21.42857142857143 00.30	 Player 1 - Serral (Zerg) - Stats Update
22.05357142857143 00.30	 Player 1 - Serral (Zerg) - Unit born Drone [3CC0001]
22.05357142857143 00.30	 Player 1 - Serral (Zerg) - Unit Larva [3740001] type changed to Larva
22.05

In [17]:
output_df


,game_number,opponent_race,game_length_seconds_max_600,larva_time,larva_amount
0,1,Protoss,600,10.223214,1
1,1,Protoss,600,20.133929,1
2,1,Protoss,600,30.044643,1
3,1,Protoss,600,39.955357,1
4,1,Protoss,600,49.866071,1
...,...,...,...,...,...
237,1,Protoss,600,576.741071,1
238,1,Protoss,600,576.785714,2
239,1,Protoss,600,577.142857,3
240,1,Protoss,600,579.375000,2
